# 1. Setup & Load Data
Load data hasil cleaning dari tahap EDA.

---



In [1]:
from google.colab import drive
drive.mount('/content/drive')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.preprocessing import MinMaxScaler
import os
import pickle

BASE_PATH = '/content/drive/MyDrive/stock_prediction-lstm-tft'
RAW_PATH  = f'{BASE_PATH}/02_data/raw'
CLEAN_PATH = f'{BASE_PATH}/02_data/clean'
COLAB_PATH = f'{BASE_PATH}/03_colab'
MODEL_DATA_PATH = f'{BASE_PATH}/04_model/data'

df = pd.read_csv(f'{CLEAN_PATH}/lq45_cleaned.csv')
df['price_date'] = pd.to_datetime(df['price_date'])
df = df.sort_values(by=['symbol', 'price_date']).reset_index(drop=True)
print("Shape:", df.shape)
df.head()

Mounted at /content/drive
Shape: (7235, 9)


,id,symbol,company_name,open_price,high_price,low_price,close_price,volume,price_date
0,2895,ASII.JK,Astra International,4447.92,4447.92,4351.57,4415.80,15008600.0,2020-01-02
1,2896,ASII.JK,Astra International,4447.92,4463.98,4383.69,4463.98,19068800.0,2020-01-03
2,2897,ASII.JK,Astra International,4431.86,4431.86,4335.52,4335.52,22261900.0,2020-01-06
3,2898,ASII.JK,Astra International,4367.63,4383.69,4287.34,4351.57,27963000.0,2020-01-07
4,2899,ASII.JK,Astra International,4303.40,4383.69,4303.40,4351.57,15150400.0,2020-01-08


# 2. Feature Engineering

Penambahan indikator teknikal sebagai fitur tambahan untuk mendukung model,
khususnya TFT yang mampu menangani input heterogen.

Indikator yang ditambahkan:
- **MA7 & MA30**: Moving Average 7 dan 30 hari untuk menangkap tren jangka pendek dan menengah
- **RSI**: Relative Strength Index untuk mengukur momentum harga
- **Volume**: tetap dipertahankan sebagai sinyal kekuatan pergerakan pasar

---



In [2]:
def compute_rsi(series, period=14):
    delta = series.diff()
    gain = delta.where(delta > 0, 0)
    loss = -delta.where(delta < 0, 0)
    avg_gain = gain.rolling(window=period).mean()
    avg_loss = loss.rolling(window=period).mean()
    rs = avg_gain / avg_loss
    return 100 - (100 / (1 + rs))

results = []

for symbol in df['symbol'].unique():
    data = df[df['symbol'] == symbol].copy()
    data['MA7']  = data['close_price'].rolling(7).mean()
    data['MA30'] = data['close_price'].rolling(30).mean()
    data['RSI']  = compute_rsi(data['close_price'])
    results.append(data)

df = pd.concat(results).reset_index(drop=True)
df = df.dropna().reset_index(drop=True)
print("Shape setelah feature engineering:", df.shape)
df.head()

Shape setelah feature engineering: (7090, 12)


,id,symbol,company_name,open_price,high_price,low_price,close_price,volume,price_date,MA7,MA30,RSI
0,2924,ASII.JK,Astra International,3901.96,3918.02,3805.62,3837.73,51100700.0,2020-02-12,4035.010000,4324.275333,18.840593
1,2925,ASII.JK,Astra International,3837.73,3869.85,3805.62,3821.68,33021700.0,2020-02-13,3993.720000,4304.471333,19.403263
2,2926,ASII.JK,Astra International,3837.73,3918.02,3805.62,3918.02,60901000.0,2020-02-14,3966.192857,4286.272667,30.158558
3,2927,ASII.JK,Astra International,3950.14,3966.19,3885.91,3918.02,16567600.0,2020-02-17,3929.490000,4272.356000,25.423532
4,2928,ASII.JK,Astra International,3885.91,3982.25,3885.91,3918.02,27960300.0,2020-02-18,3901.962857,4257.904333,25.423532


# 3. Normalisasi Data
Normalisasi menggunakan Min-Max Scaling untuk mengubah seluruh fitur numerik
ke rentang 0–1 agar model tidak bias terhadap fitur dengan skala besar.

---



In [5]:
feature_cols = ['open_price', 'high_price', 'low_price', 'close_price', 'volume', 'MA7', 'MA30', 'RSI']

scalers = {}
df_scaled = df.copy()

for symbol in df['symbol'].unique():
    idx = df['symbol'] == symbol
    scaler = MinMaxScaler()
    df_scaled.loc[idx, feature_cols] = scaler.fit_transform(df.loc[idx, feature_cols])
    scalers[symbol] = scaler

df_scaled.head()

,id,symbol,company_name,open_price,high_price,low_price,close_price,volume,price_date,MA7,MA30,RSI
0,2924,ASII.JK,Astra International,0.414058,0.376342,0.403136,0.403023,0.138231,2020-02-12,0.419954,0.497745,0.140010
1,2925,ASII.JK,Astra International,0.399555,0.365007,0.403136,0.399286,0.078378,2020-02-13,0.409551,0.492557,0.146375
2,2926,ASII.JK,Astra International,0.399555,0.376342,0.403136,0.421716,0.170676,2020-02-14,0.402615,0.487790,0.268029
3,2927,ASII.JK,Astra International,0.424937,0.387677,0.421766,0.421716,0.023904,2020-02-17,0.393368,0.484144,0.214471
4,2928,ASII.JK,Astra International,0.410434,0.391456,0.421766,0.421716,0.061621,2020-02-18,0.386432,0.480358,0.214471


# 4. Pembentukan Sliding Window
Data disusun ke dalam format sekuensial 3D (samples, timesteps, features)
menggunakan pendekatan sliding window dengan lookback 30 hari untuk
memprediksi harga close hari berikutnya.

---



In [6]:
LOOKBACK = 30

def create_sequences(data, lookback):
    X, y = [], []
    for i in range(lookback, len(data)):
        X.append(data[i-lookback:i])
        y.append(data[i, 3])  # index 3 = close_price
    return np.array(X), np.array(y)

X_dict, y_dict = {}, {}

for symbol in df_scaled['symbol'].unique():
    data = df_scaled[df_scaled['symbol'] == symbol][feature_cols].values
    X, y = create_sequences(data, LOOKBACK)
    X_dict[symbol] = X
    y_dict[symbol] = y
    print(f"{symbol} → X: {X.shape}, y: {y.shape}")

ASII.JK → X: (1388, 30, 8), y: (1388,)
BBCA.JK → X: (1388, 30, 8), y: (1388,)
ICBP.JK → X: (1388, 30, 8), y: (1388,)
TLKM.JK → X: (1388, 30, 8), y: (1388,)
UNVR.JK → X: (1388, 30, 8), y: (1388,)


# 5. Pembagian Dataset
Dataset dibagi secara kronologis menjadi tiga bagian:
- **70%** data latih (training set)
- **15%** data validasi (validation set)
- **15%** data uji (test set)

Pembagian dilakukan tanpa pengacakan (shuffle=False) untuk menghindari data leakage.

---



In [7]:
split_dict = {}

for symbol in X_dict:
    X, y = X_dict[symbol], y_dict[symbol]
    n = len(X)
    train_end = int(n * 0.70)
    val_end   = int(n * 0.85)

    split_dict[symbol] = {
        'X_train': X[:train_end],      'y_train': y[:train_end],
        'X_val':   X[train_end:val_end], 'y_val': y[train_end:val_end],
        'X_test':  X[val_end:],        'y_test':  y[val_end:]
    }

    print(f"{symbol} → Train: {split_dict[symbol]['X_train'].shape}, "
          f"Val: {split_dict[symbol]['X_val'].shape}, "
          f"Test: {split_dict[symbol]['X_test'].shape}")

ASII.JK → Train: (971, 30, 8), Val: (208, 30, 8), Test: (209, 30, 8)
BBCA.JK → Train: (971, 30, 8), Val: (208, 30, 8), Test: (209, 30, 8)
ICBP.JK → Train: (971, 30, 8), Val: (208, 30, 8), Test: (209, 30, 8)
TLKM.JK → Train: (971, 30, 8), Val: (208, 30, 8), Test: (209, 30, 8)
UNVR.JK → Train: (971, 30, 8), Val: (208, 30, 8), Test: (209, 30, 8)


# 6. Simpan Hasil Preprocessing
Menyimpan data yang telah diproses ke direktori yang sesuai.

---



In [9]:
# Simpan df_scaled ke 02_data/clean
df_scaled.to_csv(f'{CLEAN_PATH}/lq45_scaled.csv', index=False)

# Simpan scalers dan split_data ke 03_colab
with open(f'{COLAB_PATH}/scalers.pkl', 'wb') as f:
    pickle.dump(scalers, f)

with open(f'{COLAB_PATH}/split_data.pkl', 'wb') as f:
    pickle.dump(split_dict, f)

# Simpan array numpy ke 04_model/data
for symbol in split_dict:
    sym = symbol.replace('.', '_')
    np.save(f'{MODEL_DATA_PATH}/{sym}_X_train.npy', split_dict[symbol]['X_train'])
    np.save(f'{MODEL_DATA_PATH}/{sym}_X_val.npy',   split_dict[symbol]['X_val'])
    np.save(f'{MODEL_DATA_PATH}/{sym}_X_test.npy',  split_dict[symbol]['X_test'])
    np.save(f'{MODEL_DATA_PATH}/{sym}_y_train.npy', split_dict[symbol]['y_train'])
    np.save(f'{MODEL_DATA_PATH}/{sym}_y_val.npy',   split_dict[symbol]['y_val'])
    np.save(f'{MODEL_DATA_PATH}/{sym}_y_test.npy',  split_dict[symbol]['y_test'])

print("all done!")

all done!
